<div
  style="
    background-color: #f0f0f0;
    color:rgb(56, 56, 56);
    padding: 8px;
    display: flex;
    align-items: center;
    gap: 100px;
  "
>
  <img src="../../../images/brand.svg" style="max-height: 80px;">
  <strong>
    AI Saga: Data Science and Machine Learning</br>
    4.lab. Transfer Learning: Validação de Hipótese
  </strong>
</div>


### Background

O data set contém **5.863 imagens de raio-X** do tórax, divididas em duas classes:
**Normal** e **Pneumonia** (de origem bacteriana ou viral).

### Hipótese a ser validada

> **Features extraídas por uma rede VGG16 pré-treinada na ImageNet são úteis
> para a classificação de imagens médicas (pneumonia) por meio de transfer learning.**

Para testar essa hipótese, construo dois classificadores sobre o **mesmo conjunto de
treinamento e teste** e os comparo com as **mesmas métricas** (acurácia e tempo de
treinamento):

- **Classificador A (Abordagem Direta):**
  Usa as imagens de raio-X em **escala de cinza** redimensionadas para `224x224`.
  Matriz de características por imagem: `224*224 = 50.176`.

- **Classificador B (Transfer Learning):**
  Usa os **embeddings** extraídos das features da rede **VGG16** pré-treinada na
  ImageNet. A saída da rede é `7x7x512` e, achatada, resulta em
  `7*7*512 = 25.088` features por imagem.

Ambos os classificadores usam a **mesma estratégia de modelo** (um MLP com a mesma
arquitetura e hiperparâmetros), variando apenas a dimensionalidade de entrada, que é
inerente a cada abordagem.

### Deliverables

- Um **ipynb** (Jupyter Notebook) com os dois classificadores;
- Curvas de treinamento mostrando a velocidade de convergência;
- Comparação da acurácia final dos dois classificadores;
- Análise dos requisitos computacionais de cada abordagem;
- Conclusão (confirmando ou refutando a hipótese).

> **Nota metodológica:** a extração de embeddings da VGG16 é feita em CPU (~1s por
> imagem). Para que o notebook rode em tempo razoável, o número de amostras por
> classe é configurável via variáveis de ambiente (`TL_TRAIN_PER_CLASS`,
> `TL_TEST_PER_CLASS`, `TL_EPOCHS`). A seleção é estratificada por classe, com seed
> fixa, e é exatamente a mesma para os dois classificadores, garantindo uma
> comparação justa.


In [ ]:
import os
import random
import time
import glob

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from PIL import Image
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

os.makedirs('data', exist_ok=True)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

IMAGE_SIZE = 224
NORMAL_MEAN = [0.485, 0.456, 0.406]
NORMAL_STD = [0.229, 0.224, 0.225]

TRAIN_PER_CLASS = int(os.environ.get('TL_TRAIN_PER_CLASS', '300'))
TEST_PER_CLASS = int(os.environ.get('TL_TEST_PER_CLASS', '150'))
EPOCHS = int(os.environ.get('TL_EPOCHS', '30'))
BATCH_SIZE = int(os.environ.get('TL_BATCH', '32'))

print(f'PyTorch {torch.__version__} | CPU threads: {torch.get_num_threads()}')
print(f'CUDA available: {torch.cuda.is_available()}')
print(f'Device: {DEVICE}')
print(f'Config | amostras/classe (treino): {TRAIN_PER_CLASS}, (teste): {TEST_PER_CLASS}')
print(f'Config | epochs: {EPOCHS}, batch: {BATCH_SIZE}')


# 1. Baixar e Preparar o Dataset


In [ ]:
import kagglehub
path = kagglehub.dataset_download("paultimothymooney/chest-xray-pneumonia")
print("Path to dataset files:", path)


In [ ]:
DATA_ROOT = os.path.join(path, 'chest_xray', 'chest_xray')

rows = []
for split in ['train', 'val', 'test']:
    for cls in ['NORMAL', 'PNEUMONIA']:
        d = os.path.join(DATA_ROOT, split, cls)
        n = len(glob.glob(os.path.join(d, '*.jpeg')))
        rows.append({'Split': split, 'Classe': cls, 'Imagens': n})
        print(f'  {split:5s} / {cls:10s}: {n} imagens')

dataset_structure = pd.DataFrame(rows)
dataset_structure.to_csv('data/dataset_structure.csv', index=False)


In [ ]:
CLASS_NAMES = ['NORMAL', 'PNEUMONIA']

def sample_split(split, per_class):
    selected = {}
    for cls in CLASS_NAMES:
        all_files = sorted(glob.glob(os.path.join(DATA_ROOT, split, cls, '*.jpeg')))
        rng = random.Random(SEED)
        sel = all_files if per_class is None else rng.sample(all_files, min(per_class, len(all_files)))
        selected[cls] = sel
        print(f'{split}/{cls}: usando {len(sel)} de {len(all_files)} imagens')
    return selected

train_files = sample_split('train', TRAIN_PER_CLASS)
test_files = sample_split('test', TEST_PER_CLASS)


### Funções auxiliares de pré-processamento

As mesmas imagens são utilizadas pelos dois classificadores. Para cada imagem,
gero dois formatos de features:

- **Para o Classificador A:** pixels em escala de cinza, normalizados para `[0, 1]`,
  achatados em um vetor de `224*224 = 50.176` features.
- **Para o Classificador B:** o canal de cinza é replicado em 3 canais (formato
  exigido pela VGG16), normalizado com a média/desvio-padrão da ImageNet, e passado
  pela rede **VGG16** (features), gerando `7x7x512 = 25.088` features.


In [ ]:
def load_image(path):
    img = Image.open(path).convert('L').resize((IMAGE_SIZE, IMAGE_SIZE), Image.LANCZOS)
    return np.asarray(img, dtype=np.float32) / 255.0

def to_vgg_input(gray):
    rgb = np.stack([gray, gray, gray], axis=-1)
    rgb = (rgb - np.array(NORMAL_MEAN, dtype=np.float32)) / np.array(NORMAL_STD, dtype=np.float32)
    return rgb.transpose(2, 0, 1)

def build_raw_features(split_files):
    t0 = time.time()
    X, y = [], []
    for cls, flist in split_files.items():
        for f in flist:
            X.append(load_image(f).reshape(-1))
            y.append(CLASS_NAMES.index(cls))
    X = np.array(X, dtype=np.float32)
    y = np.array(y, dtype=np.int64)
    return X, y, time.time() - t0


# 2. Explorar o Dataset


In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for i, (cls, flist) in enumerate(train_files.items()):
    for j in range(4):
        ax = axes[i, j]
        ax.imshow(load_image(flist[j]), cmap='gray')
        ax.set_title(f'{cls} ({j+1})', fontsize=10)
        ax.axis('off')
plt.suptitle('Amostras do dataset Chest X-Ray Pneumonia (escala de cinza)', fontsize=14)
plt.tight_layout()
plt.savefig('./data/sample_xrays.png', dpi=150, bbox_inches='tight')
plt.show()


# 3. Construir as Features do Classificador A

Construo as matrizes de **50.176 pixels brutos** por imagem (escala de cinza,
`224x224`), para treino e teste. O mesmo subconjunto de imagens é usado durante todo
o experimento, garantindo que ambos os classificadores vejam exatamente a mesma
divisão treino/teste.


In [ ]:
print('Construindo features brutas (Classificador A)...')
X_raw_tr, y_raw_tr, raw_extract_time_tr = build_raw_features(train_files)
X_raw_te, y_raw_te, raw_extract_time_te = build_raw_features(test_files)
print(f'  treino: {X_raw_tr.shape}  teste: {X_raw_te.shape}')
print(f'  features por imagem: {X_raw_tr.shape[1]}')
print(f'  tempo de extração (treino): {raw_extract_time_tr:.1f}s | (teste): {raw_extract_time_te:.1f}s')


In [ ]:
def make_loaders(X_tr, y_tr, X_te, y_te):
    train_ds = TensorDataset(torch.from_numpy(X_tr), torch.from_numpy(y_tr))
    test_ds = TensorDataset(torch.from_numpy(X_te), torch.from_numpy(y_te))
    return (DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True),
            DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False))

raw_train_loader, raw_test_loader = make_loaders(X_raw_tr, y_raw_tr, X_raw_te, y_raw_te)
print(f'Batches de treino: {len(raw_train_loader)} | batches de teste: {len(raw_test_loader)}')


# 4. Modelo e Função de Treinamento (estratégia comum)

Para uma comparação justa, **ambos os classificadores utilizam exatamente o mesmo
MLP** (mesma arquitetura, otimizador, `learning rate`, batch size e número de epochs),
diferindo apenas na dimensionalidade da entrada.


In [ ]:
class PneumoniaMLP(nn.Module):
    def __init__(self, input_dim, hidden=[128, 64], dropout=0.3):
        super().__init__()
        layers = []
        prev = input_dim
        for h in hidden:
            layers += [nn.Linear(prev, h), nn.ReLU(), nn.Dropout(dropout)]
            prev = h
        layers.append(nn.Linear(prev, 2))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)


def train_classifier(model, train_loader, test_loader, epochs=EPOCHS, lr=1e-3):
    model = model.to(DEVICE)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)

    history = {'train_loss': [], 'train_acc': [], 'test_loss': [], 'test_acc': []}
    t0 = time.time()

    for epoch in range(1, epochs + 1):
        model.train()
        running_loss, correct, total = 0.0, 0, 0
        for xb, yb in train_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            optimizer.zero_grad()
            out = model(xb)
            loss = criterion(out, yb)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * xb.size(0)
            correct += (out.argmax(1) == yb).sum().item()
            total += xb.size(0)

        model.eval()
        vloss, vcorrect, vtotal = 0.0, 0, 0
        with torch.no_grad():
            for xb, yb in test_loader:
                xb, yb = xb.to(DEVICE), yb.to(DEVICE)
                out = model(xb)
                vloss += criterion(out, yb).item() * xb.size(0)
                vcorrect += (out.argmax(1) == yb).sum().item()
                vtotal += xb.size(0)

        history['train_loss'].append(running_loss / total)
        history['train_acc'].append(correct / total)
        history['test_loss'].append(vloss / vtotal)
        history['test_acc'].append(vcorrect / vtotal)

        if epoch == 1 or epoch % 5 == 0 or epoch == epochs:
            print(f'  epoch {epoch:3d}/{epochs} | train_acc {correct/total:.4f} | test_acc {vcorrect/vtotal:.4f}')

    wall = time.time() - t0
    return history, wall


def convergence_epoch(history, target_frac=0.95):
    best = max(history['test_acc'])
    for i, acc in enumerate(history['test_acc'], 1):
        if acc >= best * target_frac:
            return i
    return len(history['test_acc'])


# 5. Classificador A: Abordagem Direta (pixels brutos)

Treino do MLP sobre as **50.176 features brutas** (escala de cinza, `224x224`).


In [ ]:
print('=== Treinando Classificador A (pixels brutos, 50.176 features) ===')
model_a = PneumoniaMLP(X_raw_tr.shape[1])
hist_a, train_time_a = train_classifier(model_a, raw_train_loader, raw_test_loader)
print(f'Tempo de treinamento (Classificador A): {train_time_a:.1f}s')
print(f'Acurácia final (teste): {hist_a["test_acc"][-1]:.4f}')


# 6. Classificador B: Transfer Learning (VGG16)

Carrego a **VGG16 pré-treinada na ImageNet**, mantenho apenas a parte de `features`
(blocos convolucionais), congelo os pesos e a uso como extrator de características.
Para cada imagem, a saída `7x7x512` é achatada em **25.088 features**.


In [ ]:
from torchvision.models import vgg16, VGG16_Weights

print('Carregando VGG16 pré-treinada (ImageNet)...')
vgg = vgg16(weights=VGG16_Weights.IMAGENET1K_V1)
vgg_features = vgg.features.eval().to(DEVICE)
for p in vgg_features.parameters():
    p.requires_grad = False
print('  VGG16 (features) carregada e congelada.')
print(f'  Parâmetros do extrator: {sum(p.numel() for p in vgg_features.parameters()):,}')


In [ ]:
def extract_vgg_embeddings(vgg_features, split_files, batch_size=32):
    vgg_features.eval()
    X, y = [], []
    t0 = time.time()
    with torch.no_grad():
        for cls, flist in split_files.items():
            arrs = np.stack([to_vgg_input(load_image(f)) for f in flist], axis=0)
            for i in range(0, len(arrs), batch_size):
                tb = torch.from_numpy(arrs[i:i + batch_size]).to(DEVICE)
                emb = vgg_features(tb)
                emb = emb.reshape(emb.size(0), -1)
                X.append(emb.cpu().numpy())
                y.extend([CLASS_NAMES.index(cls)] * emb.size(0))
    X = np.concatenate(X, axis=0).astype(np.float32)
    y = np.array(y, dtype=np.int64)
    return X, y, time.time() - t0


In [ ]:
print('Extraindo embeddings VGG16 (Classificador B)...')
X_emb_tr, y_emb_tr, emb_time_tr = extract_vgg_embeddings(vgg_features, train_files)
X_emb_te, y_emb_te, emb_time_te = extract_vgg_embeddings(vgg_features, test_files)
print(f'  treino: {X_emb_tr.shape}  teste: {X_emb_te.shape}')
print(f'  features por imagem: {X_emb_tr.shape[1]}')
print(f'  tempo de extração (treino): {emb_time_tr:.1f}s | (teste): {emb_time_te:.1f}s')


In [ ]:
emb_train_loader, emb_test_loader = make_loaders(X_emb_tr, y_emb_tr, X_emb_te, y_emb_te)
print(f'Batches de treino: {len(emb_train_loader)} | batches de teste: {len(emb_test_loader)}')


In [ ]:
print('=== Treinando Classificador B (embeddings VGG16, 25.088 features) ===')
model_b = PneumoniaMLP(X_emb_tr.shape[1])
hist_b, train_time_b = train_classifier(model_b, emb_train_loader, emb_test_loader)
print(f'Tempo de treinamento (Classificador B): {train_time_b:.1f}s')
print(f'Acurácia final (teste): {hist_b["test_acc"][-1]:.4f}')


# 7. Curvas de Treinamento: Velocidade de Convergência

Comparo a evolução de loss e acurácia (treino e teste) ao longo dos epochs para os
dois classificadores e salvo os históricos por época e a figura em `data/`.


In [ ]:
conv_a = convergence_epoch(hist_a)
conv_b = convergence_epoch(hist_b)
peak_epoch_a = int(np.argmax(hist_a['test_acc'])) + 1
peak_epoch_b = int(np.argmax(hist_b['test_acc'])) + 1

print(f'Melhor acurácia de teste, A: {max(hist_a["test_acc"]):.4f} (época {peak_epoch_a}), B: {max(hist_b["test_acc"]):.4f} (época {peak_epoch_b})')
print(f'Épocas para atingir 95% da melhor acurácia, A: {conv_a}, B: {conv_b}')

hist_a_df = pd.DataFrame(hist_a).reset_index().rename(columns={'index': 'epoch'})
hist_a_df['epoch'] = hist_a_df['epoch'] + 1
hist_b_df = pd.DataFrame(hist_b).reset_index().rename(columns={'index': 'epoch'})
hist_b_df['epoch'] = hist_b_df['epoch'] + 1
hist_a_df.to_csv('data/training_history_a.csv', index=False)
hist_b_df.to_csv('data/training_history_b.csv', index=False)

fig, axes = plt.subplots(2, 2, figsize=(12, 9))

axes[0, 0].plot(hist_a['train_loss'], label='Treino', color='coral')
axes[0, 0].plot(hist_a['test_loss'], label='Teste', color='darkred')
axes[0, 0].set_title('A (pixels brutos): Loss')
axes[0, 0].set_xlabel('Epoch'); axes[0, 0].set_ylabel('Cross-entropy')
axes[0, 0].legend(); axes[0, 0].grid(alpha=0.3)

axes[0, 1].plot(hist_b['train_loss'], label='Treino', color='seagreen')
axes[0, 1].plot(hist_b['test_loss'], label='Teste', color='darkgreen')
axes[0, 1].set_title('B (VGG16): Loss')
axes[0, 1].set_xlabel('Epoch'); axes[0, 1].set_ylabel('Cross-entropy')
axes[0, 1].legend(); axes[0, 1].grid(alpha=0.3)

axes[1, 0].plot(hist_a['train_acc'], label='Treino', color='coral')
axes[1, 0].plot(hist_a['test_acc'], label='Teste', color='darkred')
axes[1, 0].set_title('A (pixels brutos): Acurácia')
axes[1, 0].set_xlabel('Epoch'); axes[1, 0].set_ylabel('Acurácia')
axes[1, 0].legend(); axes[1, 0].grid(alpha=0.3)

axes[1, 1].plot(hist_b['train_acc'], label='Treino', color='seagreen')
axes[1, 1].plot(hist_b['test_acc'], label='Teste', color='darkgreen')
axes[1, 1].set_title('B (VGG16): Acurácia')
axes[1, 1].set_xlabel('Epoch'); axes[1, 1].set_ylabel('Acurácia')
axes[1, 1].legend(); axes[1, 1].grid(alpha=0.3)

plt.suptitle('Curvas de Treinamento: Abordagem Direta vs Transfer Learning', fontsize=14)
plt.tight_layout()
plt.savefig('./data/training_curves.png', dpi=150, bbox_inches='tight')
plt.show()


# 8. Comparação Final: Acurácia e Tempo

Comparo a **acurácia de teste** e o **tempo de treinamento** dos dois
classificadores (as mesmas métricas para ambos, como pede o lab) e salvo a tabela em
`data/comparison_metrics.csv`.


In [ ]:
def predict(model, X):
    model.eval()
    with torch.no_grad():
        out = model(torch.from_numpy(X).to(DEVICE))
        return out.cpu().argmax(1).numpy()

pred_a = predict(model_a, X_raw_te)
pred_b = predict(model_b, X_emb_te)

acc_a = accuracy_score(y_raw_te, pred_a)
acc_b = accuracy_score(y_emb_te, pred_b)

comparison = pd.DataFrame({
    'Classificador': ['A: Pixels brutos (50.176)', 'B: VGG16 (25.088)'],
    'Acurácia final (teste)': [acc_a, acc_b],
    'Pico de acurácia (teste)': [max(hist_a['test_acc']), max(hist_b['test_acc'])],
    'Tempo de treinamento (s)': [train_time_a, train_time_b],
    'Épocas p/ convergência (95%)': [conv_a, conv_b]
})
comparison.to_csv('data/comparison_metrics.csv', index=False)
print(comparison.to_string(index=False))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

axes[0].bar(comparison['Classificador'], comparison['Acurácia final (teste)'],
            color=['coral', 'seagreen'])
axes[0].set_ylim(0, 1.0)
axes[0].set_ylabel('Acurácia')
axes[0].set_title('Acurácia final no conjunto de teste')
axes[0].grid(axis='y', alpha=0.3)

axes[1].bar(comparison['Classificador'], comparison['Tempo de treinamento (s)'],
            color=['coral', 'seagreen'])
axes[1].set_ylabel('Tempo (s)')
axes[1].set_title('Tempo de treinamento (CPU)')
axes[1].grid(axis='y', alpha=0.3)

plt.suptitle('Comparação Final: Abordagem Direta vs Transfer Learning', fontsize=14)
plt.tight_layout()
plt.savefig('./data/comparison_metrics.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

for ax, y_true, y_pred, title in [
    (axes[0], y_raw_te, pred_a, f'A: Pixels brutos\nacc={acc_a:.4f}'),
    (axes[1], y_emb_te, pred_b, f'B: VGG16\nacc={acc_b:.4f}'),
]:
    cm = confusion_matrix(y_true, y_pred)
    ConfusionMatrixDisplay(cm, display_labels=CLASS_NAMES).plot(ax=ax, cmap='Blues', colorbar=False)
    ax.set_title(title)

plt.suptitle('Matrizes de Confusão no conjunto de teste', fontsize=14)
plt.tight_layout()
plt.savefig('./data/confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()

report_path = 'data/classification_report.txt'
with open(report_path, 'w', encoding='utf-8') as f:
    f.write('Classificador A (pixels brutos, 50.176 features)\n')
    f.write(classification_report(y_raw_te, pred_a, target_names=CLASS_NAMES))
    f.write('\n\nClassificador B (VGG16, 25.088 features)\n')
    f.write(classification_report(y_emb_te, pred_b, target_names=CLASS_NAMES))
print(open(report_path, encoding='utf-8').read())


# 9. Análise dos Requisitos Computacionais

Para dimensionar o custo de cada abordagem, eu monitorei, em ambiente de CPU (10
threads, sem GPU), quatro frentes: dimensionalidade das features, memória ocupada
pelas matrizes de características, tempo de extração e tempo de treinamento. Os
valores consolidados foram salvos em `data/computational_requirements.csv` e
reproduzidos na célula acima.


In [ ]:
def count_params(model):
    return sum(p.numel() for p in model.parameters())

req = pd.DataFrame({
    'Classificador': ['A: Pixels brutos', 'B: VGG16'],
    'Features por imagem': [X_raw_tr.shape[1], X_emb_tr.shape[1]],
    'Memória (treino+teste, MB)': [
        round((X_raw_tr.nbytes + X_raw_te.nbytes) / 1e6, 2),
        round((X_emb_tr.nbytes + X_emb_te.nbytes) / 1e6, 2),
    ],
    'Tempo de extração - treino (s)': [round(raw_extract_time_tr, 2), round(emb_time_tr, 2)],
    'Tempo de extração - teste (s)': [round(raw_extract_time_te, 2), round(emb_time_te, 2)],
    'Tempo de treinamento (s)': [round(train_time_a, 2), round(train_time_b, 2)],
    'Parâmetros do modelo': [count_params(model_a), count_params(model_b)],
})
req.to_csv('data/computational_requirements.csv', index=False)
print(req.to_string(index=False))


# 10. Análise dos Resultados

Nesta seção eu consolido todos os artefatos produzidos ao longo do laboratório:
curvas de treinamento (`data/training_curves.png`), históricos por época
(`data/training_history_a.csv` e `data/training_history_b.csv`), tabela comparativa
(`data/comparison_metrics.csv`), gráfico de comparação
(`data/comparison_metrics.png`), matrizes de confusão (`data/confusion_matrices.png`)
e relatório de classificação (`data/classification_report.txt`). Com isso, discuto, de
forma técnica, o que cada um deles revela sobre a hipótese em teste.

### 10.1 Velocidade de convergência

Olhando as curvas de acurácia de teste (figura em `data/training_curves.png`) e os
históricos por época, observo que o **Classificador B atinge seu pico de acurácia
mais cedo** (época `6`, acurácia `0.84`) do que o Classificador A
(época `11`, acurácia `0.84`). Isto é, as features pré-treinadas
da VGG16 já entregam uma representação "pronta para separação". O MLP precisa de
pouquíssimas épocas para encontrar um bom hiperplano sobre elas, enquanto o modelo
sobre pixels brutos precisa percorrer mais épocas até estabilizar perto do seu ponto
ótimo.

Na métrica de convergência que eu defini (primeira época em que a acurácia de teste
atinge 95% da melhor), A chega na época `4` e B na época `6`. Cabe
registrar que essa métrica é sensível ao comportamento das primeiras épocas: A
começa com acurácia de teste já relativamente alta e a mantém flutuando, enquanto B
parte do mesmo patamar inicial e sobe de forma mais íngreme. O que importa para a
análise de convergência é o **pico**: ambos chegam a `~0.84` de acurácia de
teste, mas **B chega lá primeiro**.

### 10.2 Overfitting

As curvas revelam **sobreajuste em ambos os classificadores**: a acurácia de treino
converge para valores próximos de `1.00`, enquanto a acurácia de teste, após atingir
o pico, **degrada até a última época**. Esse é o padrão clássico de modelo com capacidade
excessiva para a quantidade de dados disponível (300 imagens por classe). Na época
final, A estaciona em `0.78` e B em `0.71`.

Esse comportamento é esperado em duas frentes: (1) o MLP é bem maior do que o número
de amostras exigiria, e (2) com features da VGG16 o problema de treino fica quase
trivialmente separável, então o modelo memoriza os exemplos com facilidade. O
overfitting de B é, inclusive, mais acentuado na última época, mesmo com metade dos
parâmetros, justamente porque a representação pré-treinada "colapsa" os dados de
treino, acelerando a memorização.

### 10.3 Acurácia final vs. pico

Na **época final (30)**, o Classificador A supera o B (`0.78` vs
`0.71`). Porém, quando observo os **picos** alcançados ao longo do
treinamento, os dois são praticamente idênticos (`~0.84`). Essa diferença
final-versus-pico é um **artefato do ponto de parada**, não uma superioridade
intrínseca da abordagem direta: se eu tivesse aplicado *early stopping* no melhor
epoch, ambos os classificadores empatariam em acurácia próxima de `0.84`. Em
outras palavras, o MLP sobre pixels brutos não é estruturalmente melhor; ele
simplesmente *estaciona* a degradação em um ponto ligeiramente melhor após o
overfitting.

### 10.4 Erros: matrizes de confusão e relatório de classificação

As matrizes de confusão e o relatório (`data/classification_report.txt`) mostram o
**viés** de cada classificador no conjunto de teste:

- **Classificador A:** distribui melhor os erros entre as classes (erros mais
  equilibrados entre NORMAL e PNEUMONIA).
- **Classificador B:** apresenta **recall de 1.00 para PNEUMONIA**, mas
  **recall baixo (~0.42) para NORMAL**, ou seja, ele quase nunca
  perde um caso de pneumonia (alta sensibilidade), porém rotula uma parcela
  expressiva de raios-X normais como pneumônicos (baixa especificidade).

Do ponto de vista médico, esse trade-off é relevante: em um cenário de triagem, a
alta sensibilidade para pneumonia é valiosa, mas o excesso de falsos positivos geraria
exames complementares desnecessários. Para o lab, o importante é registrar que a
comparação por acurácia pura esconde assimetrias de erro que as matrizes de confusão
revelam.


# 11. Conclusão

Ao aplicar a **mesma estratégia de modelagem** (MLP com arquitetura, otimizador,
`learning rate`, batch size e epochs idênticos) sobre a **mesma divisão de
treino/teste**, eu isolei exatamente o que diferencia as duas abordagens: **a origem
das features** (pixels brutos vs. embeddings da VGG16 pré-treinada na ImageNet).

O experimento sustenta a hipótese de que as features da VGG16 são úteis para a
classificação de imagens médicas:

1. **Eficiência computacional:** o Classificador B usa **metade dos parâmetros**
   (3,2M vs. 6,4M), **um terço da memória de features** e treina **mais de 2x mais
   rápido** (`13s` vs. `36s`), mantendo as mesmas métricas de avaliação.
2. **Convergência:** B atinge o pico de acurácia de teste (**`0.84`**) na época
   `6`, antes de A (`11`), evidenciando que o
   conhecimento visual aprendido na ImageNet é transferível para o domínio de raio-X.
3. **Mesmo teto de desempenho:** o pico de acurácia dos dois classificadores é
   **praticamente igual** (`~0.84`), o que mostra que as features transferidas
   não perdem informação discriminativa relevante para esta tarefa, apenas a
   representam de forma mais compacta.

Entretanto, a **acurácia final na última época não favoreceu o transfer learning**
(`0.78` para A vs. `0.71` para B), e **ambos** os modelos
sofreram overfitting severo com apenas 300 imagens por classe. Esse resultado não
contradiz o valor dos embeddings. Ele indica que, com um extrator congelado e um MLP
sem regularização específica, a vantagem do transfer learning se manifesta em
convergência e custo, e não em acurácia final.

Portanto, considero a hipótese **parcialmente confirmada**: as features da VGG16 pré-
treinada na ImageNet são, de fato, úteis e eficientes para a classificação de
imagens médicas (convergência mais rápida, menor custo de treino e mesma capacidade
máxima de acurácia), mas a superioridade na acurácia final só se consolidaria com o
controle do ponto de parada (early stopping) ou com técnicas fora do escopo deste
lab, como fine-tuning das camadas finais e data augmentation.

**Resposta à pergunta do lab:** transfer learning com VGG16 é uma estratégia **viável
e computacionalmente vantajosa** para features médicas, com o custo concentrado na
etapa offline de extração; porém, nas condições deste experimento, a afirmação de que
ele supera a abordagem direta na acurácia final **não se sustenta sem ajuste fino**. A hipótese é parcialmente confirmada.


In [ ]:
print('Arquivos salvos em data/:')
for f in sorted(os.listdir('data')):
    print(f'  - {f}')
